# Corrected Google Colab Demo: Small Amharic Language Model from Hugging Face

This notebook demonstrates how to load a small Amharic instruction model from Hugging Face, prompt it in Amharic, and compare outputs.

**Recommended runtime:** Google Colab GPU  
Go to **Runtime → Change runtime type → T4 GPU**.

The previous `bigscience/bloomz-560m` output was not good for Amharic. This notebook instead uses an Amharic-specific model.

In [1]:
# ============================================================
# 1. Install libraries
# ============================================================
!pip install -q -U transformers accelerate sentencepiece safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 60.2 MB/s eta 0:00:00


In [1]:
# ============================================================
# 2. Import libraries and check GPU
# ============================================================
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"   # helps avoid some Colab download issues

import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No GPU detected. The code will still run, but it may be slower.")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory GB: 14.56


In [2]:
# ============================================================
# 3. Choose the model
# ============================================================
# Better quality, still small enough for Colab T4:
MODEL_ID = "rasyosef/Llama-3.2-1B-Amharic-Instruct"

# Faster fallback if the 1B model fails or the class has weak machines:
FALLBACK_MODEL_ID = "rasyosef/Llama-3.2-180M-Amharic-Instruct"

print("Main model:", MODEL_ID)
print("Fallback model:", FALLBACK_MODEL_ID)

Main model: rasyosef/Llama-3.2-1B-Amharic-Instruct
Fallback model: rasyosef/Llama-3.2-180M-Amharic-Instruct


In [3]:
# ============================================================
# 4. Load tokenizer and model
# ============================================================
def load_model(model_id):
    print(f"Loading: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )

    model.eval()
    print("Loaded successfully.")
    return tokenizer, model


try:
    tokenizer, model = load_model(MODEL_ID)
    ACTIVE_MODEL = MODEL_ID
except Exception as e:
    print("The 1B model did not load. Error:")
    print(e)
    print("\nTrying the 180M fallback model...")
    tokenizer, model = load_model(FALLBACK_MODEL_ID)
    ACTIVE_MODEL = FALLBACK_MODEL_ID

print("Active model:", ACTIVE_MODEL)

Loading: rasyosef/Llama-3.2-1B-Amharic-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/928 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/19.2M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

Loaded successfully.
Active model: rasyosef/Llama-3.2-1B-Amharic-Instruct


In [4]:
# ============================================================
# 5. Helper functions for Amharic prompting
# ============================================================
def amharic_ratio(text):
    """Return the approximate ratio of Ethiopic-script characters in the output."""
    chars = [c for c in text if not c.isspace()]
    if not chars:
        return 0.0
    amharic_chars = re.findall(r"[\u1200-\u137F]", text)
    return len(amharic_chars) / len(chars)


def clean_answer(text):
    """Remove extra chat markers or repeated labels if they appear."""
    bad_markers = [
        "<|im_start|>", "<|im_end|>", "<|endoftext|>",
        "user", "assistant", "ጥያቄ:", "መልስ:"
    ]
    for marker in bad_markers:
        text = text.replace(marker, " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def ask_amharic(
    question,
    max_new_tokens=180,
    temperature=0.35,
    top_p=0.90,
    repetition_penalty=1.12
):
    """
    Ask the model a question in Amharic.
    Lower temperature gives more stable classroom output.
    """

    classroom_instruction = f"""
እባክህ በንጹህ እና ቀላል አማርኛ መልስ።
መልሱ ለተማሪዎች ትምህርታዊ መሆን አለበት።
ነጥቦች ከተጠየቁ በቁጥር ዘርዝር።
የማታውቀውን አትፍጠር።

ጥያቄ: {question}
""".strip()

    messages = [
        {"role": "user", "content": classroom_instruction}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens, not the original prompt
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
    answer = clean_answer(answer)

    ratio = amharic_ratio(answer)
    if ratio < 0.35:
        print("WARNING: Output is not mostly Amharic. Try:")
        print("- Runtime → Change runtime type → GPU")
        print("- Use the 1B model instead of the 180M fallback")
        print("- Lower temperature to 0.2")
        print("- Restart runtime and run all cells again")

    return answer

In [5]:
# ============================================================
# 6. Test 1: GitHub question in Amharic
# ============================================================
question = "GitHub ለተማሪዎች ምን ጥቅም አለው? በ5 ነጥቦች ግለጽ።"

answer = ask_amharic(question)
print("ጥያቄ:", question)
print("መልስ:")
print(answer)

ጥያቄ: GitHub ለተማሪዎች ምን ጥቅም አለው? በ5 ነጥቦች ግለጽ።
መልስ:
1. የክፍል ትምህርት፡ GitHub ተማሪዎች ከግል ኮምፒዩተሮች ጋር እንዲገናኙ፣ ፕሮግራሞችን እንዲያዘጋጁ ወይም ከሌሎች የኮምፒዩተር ፕሮግራሞች ጋር እንዲተባበሩ ያስችላቸዋል። 2. የመረጃ ትንተና፡- GitHub ተጠቃሚዎች መረጃን እንዲረዱ እና በመረጃ ላይ የተመሰረተ ውሳኔ እንዲያደርጉ የሚያስችል የላቀ የመረጃ ትንተና ስርዓት ነው። 3. የተግባር አስተዳደር፡ GitHub ተማሪዎች እንደ አስፈላጊነቱ ስልጠናን መስጠት፣ ለወደፊት ተግባሮቻቸው መመሪያዎችን ማዘጋጀት እና ግብዓቶችን መገምገም ይችላሉ። 4. የመማሪያ አካባቢ፡ GitHub ተማሪዎችን ለማቆየት እና ለመማር ምቹ የሆኑ የተለያዩ ቦታዎችን ይሰጣል። 5. የደንበኞች አገልግሎት፡ GitHub ደንበኞች በተለያዩ ቻናሎች ውስጥ እርዳታ ለመስጠት እና ጥያቄዎችን ለመመለስ እንዲሁም በማንኛውም ጊዜ ድጋፍ ለማግኘት ሊረዳቸው ይችላል።


In [7]:
# ============================================================
# 7. Test 2: Google Colab question in Amharic
# ============================================================
question = "Google Colab ምንድነው? ለAI ትምህርት ለምን ይጠቅማል?"

answer = ask_amharic(question)
print("ጥያቄ:", question)
print("መልስ:")
print(answer)

ጥያቄ: Google Colab ምንድነው? ለAI ትምህርት ለምን ይጠቅማል?
መልስ:
ለጥያቄው መልሱ የንጽህና፣ ቀለል ያለ ቋንቋ ነው።


In [9]:
# ============================================================
# 8. Test 3: Hugging Face question in Amharic
# ============================================================
question = "Hugging Face ምንድነው? ሞዴል ለመውረድ እና ለመጠቀም እንዴት ይረዳል?"

answer = ask_amharic(question)
print("ጥያቄ:", question)
print("መልስ:")
print(answer)

ጥያቄ: Hugging Face ምንድነው? ሞዴል ለመውረድ እና ለመጠቀም እንዴት ይረዳል?
መልስ:
blockchain እንደ የማሽን መማር፣ የተፈጥሮ ቋንቋ ማቀነባበሪያ (NLP)፣ የማሽን መማሪያ (ኤምኤል) እና ሌሎች በርካታ መሳሪያዎችን በመጠቀም ለመማር እና ለማሰልጠን የሚያገለግል የኮምፒዩተር ስርዓት ነው። ሞዴሉ ከመረጃው ጋር እንዲገናኝ ለማድረግ የነርቭ ኔትወርኮችን ይጠቀማል, ይህም ለንግግር ማወቂያ ተግባራት ያገለግላል. እነዚህ ስልተ ቀመሮች በስልጠና ወቅት የተጠቃሚውን ግብአት ይማራሉ እና ትንበያዎችን ወይም ውሳኔዎችን ለማድረግ ጥቅም ላይ ይውላሉ። ሞዴሉ በመረጃ ውስጥ ያሉትን ንድፎች ለመለየት እና ትንበያ ለመስጠት ይጠቅማል, ይህም ተጠቃሚዎች የበለጠ ትክክለኛ ትንበያ እንዲሰጡ ያስችላቸዋል.


In [10]:
# ============================================================
# 9. Test 4: Ask the model to explain Python code in Amharic
# ============================================================
question = '''
ይህን Python code በአማርኛ አብራራ:

for i in range(5):
    print(i)
'''

answer = ask_amharic(question, max_new_tokens=220)
print("ጥያቄ:", question)
print("መልስ:")
print(answer)

- Runtime → Change runtime type → GPU
- Use the 1B model instead of the 180M fallback
- Lower temperature to 0.2
- Restart runtime and run all cells again
ጥያቄ: 
ይህን Python code በአማርኛ አብራራ:

for i in range(5):
    print(i)

መልስ:
I would classify the given tweet as: neutral.


In [11]:
# ============================================================
# 10. Compare generation settings
# ============================================================
question = "AI ሞዴል ምንድነው? በቀላል ቃላት አብራራ።"

print("LOW TEMPERATURE: stable answer")
print(ask_amharic(question, temperature=0.2))
print("\n" + "="*80 + "\n")

print("HIGHER TEMPERATURE: more creative, but may be less accurate")
print(ask_amharic(question, temperature=0.8))

LOW TEMPERATURE: stable answer
ቀላል፡ AI ሞዴልን ለማሰልጠን፣ ለመገምገም ወይም ለመተንተን የሚያገለግል የኮምፒዩተር ፕሮግራም ነው።


HIGHER TEMPERATURE: more creative, but may be less accurate
መማር ለፈጠራው ብቁ ለመሆን በቂ አይደለም.


In [ ]:
# ============================================================
# 11. Optional: Simple classroom chat loop
# ============================================================
# Run this cell and type Amharic questions.
# Type stop to finish.

while True:
    q = input("ጥያቄ ያስገቡ / type stop: ")
    if q.lower().strip() in ["stop", "exit", "quit"]:
        break
    print("\nመልስ:")
    print(ask_amharic(q))
    print("\n" + "-"*80 + "\n")

ጥያቄ ያስገቡ / type stop: ሰዉ ምንድን ነዉ?

መልስ:
መልስ፡ ተማሪ

--------------------------------------------------------------------------------



## Troubleshooting

If you still see strange text such as `Çàê...`:

1. Restart the Colab runtime.
2. Make sure the model is **not** `bigscience/bloomz-560m`.
3. Use `rasyosef/Llama-3.2-1B-Amharic-Instruct`.
4. Use GPU runtime if available.
5. Lower `temperature` to `0.2`.
6. Use clear Amharic instructions and ask for numbered answers.